# Vision2Slope Tutorial

This tutorial demonstrates how to use Vision2Slope for road slope analysis from street view images.


## 1. Installation

Install Vision2Slope from the local source so you can run the tutorial.


In [ ]:
# Install Vision2Slope package from source
import subprocess
import sys

# Navigate to package directory and install in editable mode
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "../src/Vision2Slope"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Vision2Slope package installed successfully!")
else:
    print(f"✗ Installation error: {result.stderr}")


## 2. Environment Setup and Imports

First, import the necessary libraries and modules.

In [ ]:
# Import required libraries
from pathlib import Path

# Import Vision2Slope modules
from vision2slope import (
    PipelineConfig,
    VisualizationConfig,
    ProcessingConfig,
    Vision2SlopePipeline
)

print("✓ All libraries imported successfully!")
print(f"Working directory: {Path.cwd()}")


## 3.1. Basic Usage: Processing Panoramic Images

Convert panoramic street view images to perspective views and analyze them.


## 3.1.1 (Optional) Download Mapillary Panoramas

Download panoramic street view images from Mapillary for analysis.

**Note:** Requires a Mapillary API key (register at https://www.mapillary.com/) and the `zensvi` package, installed separately with `pip install zensvi` (Python 3.9–3.12 only).


In [ ]:
# Set your Mapillary API key (get it from https://www.mapillary.com/)
mly_api_key = "YOUR_MAPILLARY_API_KEY_HERE"  # Replace with your actual API key

# Configure download options
from zensvi.download import MLYDownloader

downloader = MLYDownloader(mly_api_key=mly_api_key)

# Example: Download panoramic images from San Francisco
downloader.download_svi(
    "sf",
    lat=37.797423890238,
    lon=-122.44403351517,
    buffer=5,
    resolution=2048,
    image_type="pano"  # 'flat', 'pano', or 'all'
)

In [ ]:
# Configure for panoramic image processing
config_panorama = PipelineConfig(
    input_dir="sf/mly_svi/batch_1",
    output_dir="sf/mly_svi/output_pano",
    processing_config=ProcessingConfig(
        is_panorama=True,           # Enable panorama preprocessing
        panorama_fov=90,           # Field of view (degrees)
        panorama_phi=0.0,           # Vertical angle
        panorama_aspects=(10, 10),  # Aspect ratio
        panorama_show_size=100,     # Scale factor
        log_level="INFO"
    ),
    viz_config=VisualizationConfig(
        save_visualizations=True,
        save_corrected_images=True,
        save_intermediate_results=True
    )
)

print("✓ Panorama pipeline configured")
print(f"  Input:  {config_panorama.input_dir}")
print(f"  Output: {config_panorama.output_dir}")

## 3.1.2 Process Panoramic Images

In [ ]:
# Process panoramic images
pipeline_panorama = Vision2SlopePipeline(config_panorama)
results_panorama = pipeline_panorama.process_batch()

print(f"\n✓ Panorama processing completed!")
print(f"  Total images: {len(results_panorama)}")
print(f"  Successful:   {len(results_panorama[results_panorama['status'] == 'success'])}")

if len(results_panorama) > 0:
    print(f"\nResults saved to: {config_panorama.output_dir}")

## 3.2. Basic Usage - Processing Perspective Images

Process regular street view images (non-panoramic format).


In [ ]:
# Configure for processing perspective images
config_perspective = PipelineConfig(
    input_dir="input",
    output_dir="output"
)

print("✓ Configuration created for perspective images")
print(f"  Input:  {config_perspective.input_dir}")
print(f"  Output: {config_perspective.output_dir}")


In [ ]:
# Run pipeline on perspective images
# Ensure you have image files in the 'input' directory before running this cell

pipeline_perspective = Vision2SlopePipeline(config_perspective)
results_perspective = pipeline_perspective.process_batch()

print(f"\n✓ Processing completed!")
print(f"  Total images: {len(results_perspective)}")
print(f"  Successful:   {len(results_perspective[results_perspective['status'] == 'success'])}")

if len(results_perspective) > 0:
    print(f"\nResults saved to: {config_perspective.output_dir}")
